In [1]:
import pandas as pd
import numpy as np
import os

# Tự dò folder dữ liệu trên Kaggle
input_dir = "/kaggle/input/competitions"
competition_folder = None

for folder in os.listdir(input_dir):
    path = os.path.join(input_dir, folder)
    if os.path.isdir(path) and "products.csv" in os.listdir(path):
        competition_folder = folder
        break

if competition_folder is None:
    raise FileNotFoundError("Không tìm thấy folder chứa dữ liệu.")

base_path = os.path.join(input_dir, competition_folder) + "/"
print("Using:", base_path)

# Đọc dữ liệu
orders = pd.read_csv(base_path + "orders.csv")
products = pd.read_csv(base_path + "products.csv")
returns = pd.read_csv(base_path + "returns.csv")
web_traffic = pd.read_csv(base_path + "web_traffic.csv")
order_items = pd.read_csv(base_path + "order_items.csv")
customers = pd.read_csv(base_path + "customers.csv")
geography = pd.read_csv(base_path + "geography.csv")
payments = pd.read_csv(base_path + "payments.csv")
sales_train = pd.read_csv(base_path + "sales.csv")

print("Data loaded!")

Using: /kaggle/input/competitions/datathon-2026-round-1/


/tmp/ipykernel_16/493255542.py:26: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(base_path + "order_items.csv")


Data loaded!


In [ ]:
# Chuyển cột ngày sang datetime
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
returns["return_date"] = pd.to_datetime(returns["return_date"], errors="coerce")
web_traffic["date"] = pd.to_datetime(web_traffic["date"], errors="coerce")
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
sales_train["Date"] = pd.to_datetime(sales_train["Date"], errors="coerce")

**Câu 1** 

In [ ]:
o1 = orders.copy()
o1 = o1.sort_values(["customer_id", "order_date"])

o1 = o1.groupby("customer_id", group_keys=False).filter(lambda x: len(x) > 1)
o1 = o1[o1["order_status"] == "delivered"].copy()

o1["prev_order_date"] = o1.groupby("customer_id")["order_date"].shift(1)
o1["days_gap"] = (o1["order_date"] - o1["prev_order_date"]).dt.days

q1_result = o1["days_gap"].median()
print(q1_result)

175.0


**Câu 2**

In [4]:
products["gross_margin"] = (products["price"] - products["cogs"]) / products["price"]

q2_result = (
    products.groupby("segment", as_index=False)["gross_margin"]
    .mean()
    .sort_values("gross_margin", ascending=False)
)

print(q2_result)

       segment  gross_margin
6     Standard      0.313442
5      Premium      0.285377
1  All-weather      0.284176
0   Activewear      0.265600
4  Performance      0.263650
2     Balanced      0.258038
7       Trendy      0.240758
3     Everyday      0.236343


**Câu 3**

In [5]:
q3_result = (
    returns.merge(products, on="product_id", how="inner")
    .query("category == 'Streetwear'")
    .groupby("return_reason")
    .size()
    .sort_values(ascending=False)
)

print(q3_result)

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
dtype: int64


**Câu 4**

In [6]:
q4_result = (
    web_traffic.groupby("traffic_source", as_index=False)["bounce_rate"]
    .mean()
    .sort_values("bounce_rate")
)

print(q4_result)

   traffic_source  bounce_rate
1  email_campaign     0.004458
5    social_media     0.004476
3     paid_search     0.004478
4        referral     0.004499
2  organic_search     0.004504
0          direct     0.004511


**Câu 5**

In [7]:
total_rows = len(order_items)
promo_rows = order_items["promo_id"].notna().sum()
promo_percentage = (promo_rows / total_rows) * 100

q5_result = pd.DataFrame({
    "total_rows": [total_rows],
    "promo_rows": [promo_rows],
    "promo_percentage": [promo_percentage]
})

print(q5_result)

   total_rows  promo_rows  promo_percentage
0      714669      276316         38.663493


**Câu 6**

In [8]:
df6 = customers[customers["age_group"].notna()].merge(orders, on="customer_id", how="inner")

q6_result = (
    df6.groupby("age_group")
    .agg(
        total_orders=("customer_id", "size"),
        total_customers=("customer_id", "nunique")
    )
    .assign(avg_orders_per_customer=lambda x: x["total_orders"] / x["total_customers"])
    .sort_values("avg_orders_per_customer", ascending=False)
)

print(q6_result)

           total_orders  total_customers  avg_orders_per_customer
age_group                                                        
55+               72760            10010                 7.268731
45-54            124138            17193                 7.220264
35-44            170368            23642                 7.206159
25-34            190622            26802                 7.112230
18-24             89057            12599                 7.068577


**Câu 7**

In [9]:
q7_result = (
    orders.merge(order_items, on="order_id", how="inner")
    .merge(geography, on="zip", how="inner")
)

# Giữ đúng logic như code R của bạn: quantity * unit_price
q7_result["line_revenue"] = q7_result["quantity"] * q7_result["unit_price"]

q7_result = (
    q7_result.groupby("region", as_index=False)["line_revenue"]
    .sum()
    .sort_values("line_revenue", ascending=False)
)

print(q7_result)

    region  line_revenue
1     East  7.637533e+09
0  Central  4.941908e+09
2     West  3.851035e+09


**Câu 8**

In [10]:
q8_result = (
    orders[orders["order_status"] == "cancelled"]
    .groupby("payment_method")
    .size()
    .sort_values(ascending=False)
)

print(q8_result)

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
dtype: int64


**Câu 9**

In [11]:
sold_by_size = (
    order_items.merge(products, on="product_id", how="inner")
    .groupby("size")
    .size()
    .reset_index(name="total_sold")
)

returned_by_size = (
    returns.merge(products, on="product_id", how="inner")
    .groupby("size")
    .size()
    .reset_index(name="total_returned")
)

q9_result = (
    sold_by_size.merge(returned_by_size, on="size", how="left")
    .fillna({"total_returned": 0})
)

q9_result["return_rate"] = q9_result["total_returned"] / q9_result["total_sold"]
q9_result = q9_result[q9_result["size"].isin(["S", "M", "L", "XL"])].sort_values("return_rate", ascending=False)

print(q9_result)

  size  total_sold  total_returned  return_rate
2    S      172042            9723     0.056515
0    L      173174            9741     0.056250
1    M      176428            9820     0.055660
3   XL      193025           10655     0.055200


**Câu 10**

In [12]:
q10_result = (
    payments.groupby("installments", as_index=False)["payment_value"]
    .mean()
    .sort_values("payment_value", ascending=False)
)

print(q10_result)

   installments  payment_value
3             6   24446.654403
2             3   24399.635486
4            12   24245.772694
0             1   24113.274166
1             2     708.473729


**Tổng hợp kết quả**

In [13]:
answers = {
    "Q1": q1_result,
    "Q2": q2_result.iloc[0]["segment"],
    "Q3": q3_result.index[0],
    "Q4": q4_result.iloc[0]["traffic_source"],
    "Q5": promo_percentage,
    "Q6": q6_result.index[0],
    "Q7": q7_result.iloc[0]["region"],
    "Q8": q8_result.index[0],
    "Q9": q9_result.iloc[0]["size"],
    "Q10": q10_result.iloc[0]["installments"],
}

for k, v in answers.items():
    print(k, "->", v)

Q1 -> 175.0
Q2 -> Standard
Q3 -> wrong_size
Q4 -> email_campaign
Q5 -> 38.663493169565214
Q6 -> 55+
Q7 -> East
Q8 -> credit_card
Q9 -> S
Q10 -> 6.0


**Tổng hợp đáp án**

Q1: 175.0          -> C 

Q2: Standard       -> D

Q3: wrong_size     -> B

Q4: email_campaign -> C

Q5: 39%            -> C

Q6: 55+            -> A

Q7: East           -> C 

Q8: credit_card    -> A

Q9: S              -> A 

Q10: 6 kỳ          -> C 